In [0]:
%pip install confluent-kafka

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install pymongo


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from confluent_kafka import Consumer, KafkaException
from pymongo import MongoClient
import json

#Kafka Configuration
KAFKA_BOOTSTRAP_SERVERS = "" #use own
KAFKA_API_KEY = "" #use own
KAFKA_API_SECRET = "" #use own

KAFKA_TOPIC = "txndata"
KAFKA_GROUP_ID = "fraud-detection-group"

#MongoDB Configuration
MONGO_URI = "" #use own
MONGO_DB = "txn_db"
MONGO_COLLECTION1 = "fraud_alerts"
MONGO_COLLECTION2 = "non_fraud"

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[MONGO_COLLECTION1]
collection_non_fraud = db[MONGO_COLLECTION2]

#fuction fraud or not fraud:
def if_fraudulent(transaction):
    fraud_score = transaction["amount"]/500
    return fraud_score > 0.8

#Kafka Consumer
conf = {
    'bootstrap.servers': KAFKA_BOOTSTRAP_SERVERS,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': KAFKA_API_KEY,
    'sasl.password': KAFKA_API_SECRET,
    'group.id': KAFKA_GROUP_ID,
    'auto.offset.reset': 'earliest'
}

consumer = Consumer(conf)
consumer.subscribe([KAFKA_TOPIC])

print(f"listning for the msg on kafka topic {KAFKA_TOPIC}")

try:
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            print(f"kaafka error not able to get the msg {msg.error()}")
            continue
        transaction = json.loads(msg.value().decode('utf-8'))
        print(f"recieved msg from kafka is {transaction}")

        if if_fraudulent(transaction):
            print("fraud detected",transaction)
            collection.insert_one(transaction)

        else:
            collection_non_fraud.insert_one(transaction)

except Exception as e:
    print(e)
finally:
    consumer.close()


listning for the msg on kafka topic txndata
recieved msg from kafka is {'transaction_id': 'bced737c-ba67-4f21-92fa-fd7b368518b5', 'timestamp': 1760769570, 'user_id': 86842, 'amount': 4099.75, 'transaction_type': 'purchase', 'location': 'Port Matthew', 'merchant': 'Ward, Gonzalez and Campbell', 'card_number': '3569292038684619'}
fraud detected {'transaction_id': 'bced737c-ba67-4f21-92fa-fd7b368518b5', 'timestamp': 1760769570, 'user_id': 86842, 'amount': 4099.75, 'transaction_type': 'purchase', 'location': 'Port Matthew', 'merchant': 'Ward, Gonzalez and Campbell', 'card_number': '3569292038684619'}


%6|1760769581.814|GETSUBSCRIPTIONS|rdkafka#consumer-3| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to qev6RTU0SY+uAUGuJ4ZFhA


recieved msg from kafka is {'transaction_id': '446b784b-52b1-4d8c-9805-b4c78bd61190', 'timestamp': 1760769580, 'user_id': 72497, 'amount': 4097.49, 'transaction_type': 'withdrawal', 'location': 'East Jonathanberg', 'merchant': 'Brooks Ltd', 'card_number': '3552227948356899'}
fraud detected {'transaction_id': '446b784b-52b1-4d8c-9805-b4c78bd61190', 'timestamp': 1760769580, 'user_id': 72497, 'amount': 4097.49, 'transaction_type': 'withdrawal', 'location': 'East Jonathanberg', 'merchant': 'Brooks Ltd', 'card_number': '3552227948356899'}
recieved msg from kafka is {'transaction_id': 'd7d52e34-967e-4e95-85d2-92841b836bae', 'timestamp': 1760769581, 'user_id': 29693, 'amount': 3600.64, 'transaction_type': 'purchase', 'location': 'Coreytown', 'merchant': 'Taylor LLC', 'card_number': '3552258954637872'}
fraud detected {'transaction_id': 'd7d52e34-967e-4e95-85d2-92841b836bae', 'timestamp': 1760769581, 'user_id': 29693, 'amount': 3600.64, 'transaction_type': 'purchase', 'location': 'Coreytown', '

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can